# <font color='Green'> Ensemble - Project </font>

### Problem Statement:
A telecom company wants to use their historical customer data to predict behaviour to retain customers. You can analyse all relevant customer data and develop focused customer retention programs. 

### Objective:
Build a model that will help to identify the potential customers who have a higher probability to churn. This help the company to understand the pinpoints and patterns of customer churn and will increase the focus on strategising customer retention. 

## 1. Import and warehouse data:

In [ ]:
import pandas as pd
import numpy as np
# we are going with direct import method
df = pd.read_csv('../input/telecom-users-dataset/telecom_users.csv')

In [ ]:
df.head()

In [ ]:
df.info()

#### Feature details
• Customers who left within the last month – the column is called Churn   
 • Services that each customer has signed up for – phone, multiple lines, internet, online security, online backup, device protection, tech support, and streaming TV and movies  
 • Customer account information – how long they’ve been a customer, contract, payment method, paperless billing, monthly charges, and total charges  
 • Demographic info about customers – gender, age range, and if they have partners and dependents 

## 2. Data cleansing

### Missing value treatment 

In [ ]:
# Missing Values
df.isna().sum()

**No missing values found in the dataset**

### Convert categorical attributes to continuous using relevant functional knowledge 

In [ ]:
df.head().T

In [ ]:
df.info()

**Feature Total Charges is contineous value. however it is shown as Objec (ie., "string"). we will check and update it accordingly**

In [ ]:
#Function to convert the Total column to Contineous feature
def convert_to_contineous(feature):
    df[feature]=pd.to_numeric(df[feature], errors='coerce')

In [ ]:
convert_to_contineous('TotalCharges')

In [ ]:
df.info()

In [ ]:
df.isna().sum()

There are 10 null values in Total Charges after converted the value to Integer. this is because the value may contain the empty space in it. we can consider dropping the null value as the count is less than 1% of total size of dataset. 

In [ ]:
df=df.dropna(axis=0)

### Drop attribute/s if required using relevant functional knowledge 

In [ ]:
# CustomerID is the id of the customer with corresponding details.
#this information may not be requried for analysis and modeling as the customerID will be all unique values.
# so we can drop the features
df.drop(['customerID','Unnamed: 0'], axis=1, inplace=True)

In [ ]:
df.head()

## 3. Data analysis & visualisation

###  Perform detailed statistical analysis on the data

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Feature SeniorCitizen is numeric categorical values. it contains 1 and 0. however, for EDA purpose let us convert the feature to categorical.
df['SeniorCitizen']=df['SeniorCitizen'].astype('category')

In [ ]:
df.describe().T

## Univariated Analysis

### Contineous Features

In [ ]:
int_feat = df.select_dtypes(exclude=['object','category']).columns
fig, ax = plt.subplots(nrows=2, ncols = 2, figsize=(15,8), constrained_layout=True)
ax=ax.flatten()
for c,i in enumerate(int_feat):
    sns.histplot(df[i], ax=ax[c], bins=10)
    ax[c].set_title(i)

### Observations:
1. Tenure is ranging between 1 to 70 year. most prefered tenure are less than 5 and 70 years.
2. Monthly charges raning between 20 to 120. most customers are in 20-30 range
3. Total Charges - most customers total charges are between 5000

In [ ]:
cat_cols = df.select_dtypes(include=['object','category']).columns
fig, ax = plt.subplots(nrows=5, ncols=4, figsize=(15,15), constrained_layout=True)
ax=ax.flatten()
for x,i in enumerate(cat_cols):
    sns.countplot(x=df[i], ax=ax[x])

### Observations:
1. Genere: There is no much difference in Genere base customer. very few count differences.
2. Senior Citizers: comparitively less count of senior citizen as customer. 
3. Partner - Feature contains more or less same size of counts.   
Let do further analysis using multivariated analysis which helps for modeling

## Bivariated Analysis

In [ ]:
# Churn is the Target variable, let us to the analysis with the Target variable
fig, ax = plt.subplots(nrows=2, ncols = 2, figsize=(15,8), constrained_layout=True)
ax=ax.flatten()
for c,i in enumerate(int_feat):
    sns.histplot(data=df, x=i, ax=ax[c], bins=10, hue='Churn', kde=True)
    ax[c].set_title(i)

### Observations:
1. Tenure has opposite effect on Churn "Yes" or "No". Churn customers have minimum tenure, where in no churn customers prefer longer tenure
2. Monthly Charges &  Total charges have no visible difference apart from the imbalance in the data.

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols = 2, figsize=(15,8), constrained_layout=True)
ax=ax.flatten()
for c,i in enumerate(int_feat):
    sns.boxplot(data=df, x=i, ax=ax[c], y='Churn')
    ax[c].set_title(i)

There when we check the outlier for whole data there is no much outliers.however, when we consider doing the multivariated analysis, there are some seen outliers for the samples between churn and non churn. Since the objective of the project is to use the Ensemble methods, the Ensemble methods have no impact on outliers. so, outlier treatment is not required here

In [ ]:
cat_cols = df.select_dtypes(include=['object','category']).columns.to_list()
cat_cols.remove('Churn')
fig, ax = plt.subplots(nrows=4, ncols=4, figsize=(15,15), constrained_layout=True)
ax=ax.flatten()
for x,i in enumerate(cat_cols):
    sns.countplot(data=df,x=i, ax=ax[x], hue='Churn')

The sample differnce in Churn and non churn for the categorical features have no much difference appart from the imbalance in the dataset

In [ ]:
fig, ax = plt.subplots(ncols=2, nrows=2, figsize=(12,10))
ax[0,0].pie(df['gender'].value_counts(), autopct='%.2f%%', labels=df['gender'].unique())
ax[0,0].set_title('Telecom users by Gender')
ax[0,1].pie(df['SeniorCitizen'].value_counts(), autopct='%.2f%%', labels=['Non-Senior Citizen','Senior Citizer'])
ax[0,1].set_title('Telecom users by Gender')
ax[1,1].pie(df[df['SeniorCitizen']==1]['gender'].value_counts(), autopct='%.2f%%', labels=['Male','Female'])
ax[1,1].set_title('Out of Senior Citizer how many are Male & Female')
ax[1,0].pie(df[df['SeniorCitizen']==0]['gender'].value_counts(), autopct='%.2f%%', labels=['Male','Female'])
ax[1,0].set_title('Out of Non-Senior Citizer how many are Male & Female')
plt.show()

In [ ]:
fig, ax=plt.subplots(ncols=3,nrows=1, figsize=(12,10))
j=0
fig.suptitle('Gender usage of Intenet types')
for i in df['InternetService'].unique():
    ax[j].pie(df[df['InternetService']==i]['gender'].value_counts(),  labels=['Male','Female'], autopct='%.2f%%');
    ax[j].set_title(i)
    j=j+1

In [ ]:
sns.countplot(x=df['InternetService'], hue=df['Churn'])

People using Fiberoptics seems to have impact on customer Churn value. however, Hypothesis will help us to prove this right

In [ ]:
sns.jointplot(data=df,x='MonthlyCharges',y='TotalCharges', kind='kde', hue='Churn')

In [ ]:
sns.scatterplot(data=df, x='MonthlyCharges', y='TotalCharges', hue='Churn', alpha=0.5)

We can't see clear cluster or liner relationship in the datasets, hense Tree based algorithm might work well. 

## Multivariated Analysis

In [ ]:
sns.pairplot(df,hue='Churn', corner=True )

In [ ]:
sns.heatmap(df.corr(), annot=True)

There is no multicolunery seen in the dataset for numerical features

# Data pre-processing:

There are lot of Yes/No values, let us replace it with 1 or 0. assumption in bold

    PhoneService - is the telephone service connected (Yes, No) - (1,0)
    MultipleLines - are multiple phone lines connected (Yes, No, No phone service) - (1,0,0)
    InternetService - client's Internet service provider (DSL, Fiber optic, No) -(2,1,0)
    OnlineSecurity - is the online security service connected (Yes, No, No internet service)-(1,0,0)
    OnlineBackup - is the online backup service activated (Yes, No, No internet service)-(1,0,0)
    DeviceProtection - does the client have equipment insurance (Yes, No, No internet service)-(1,0,0)
    TechSupport - is the technical support service connected (Yes, No, No internet service)-(1,0,0)
    StreamingTV - is the streaming TV service connected (Yes, No, No internet service)-(1,0,0)
    StreamingMovies - is the streaming cinema service activated (Yes, No, No internet service)-(1,0,0)
    Contract - type of customer contract (Month-to-month, One year, Two year) - (month-to-month - 1, One Year - 12, Two Year = 24)
    PaperlessBilling - whether the client uses paperless billing (Yes, No) - (1,0)
    PaymentMethod - payment method (Electronic check, Mailed check, Bank transfer (automatic), Credit card (automatic))
    MonthlyCharges - current monthly payment
    TotalCharges - the total amount that the client paid for the services for the entire time
    Churn - whether there was a churn (Yes or No) - (0,1)

In [ ]:
# Feature engineering - convert the object features to integer based on the category
df=df.replace('Yes',1)
df=df.replace('No',0)
df=df.replace('No internet service',0)
df=df.replace('No phone service',0)
df=df.replace('Fiber optic',2)
df=df.replace('DSL',1)
df=df.replace('Male',1)
df=df.replace('Female',0)
df=pd.get_dummies(data=df, columns=['Contract','PaymentMethod'],drop_first=True )

In [ ]:
df

In [ ]:
#Segregate predictors vs target attributes 
X=df.drop(['Churn'], axis=1)
y=df['Churn']

In [ ]:
y.value_counts()

In [ ]:
1587/4389*100

There is imbalance in Target variable with almost 64% data in 0 and 36% data in 1. Let us try to undersample it using sample option in Padas

### Handling Imbalance in the dataset using Pandas sample function

In [ ]:
sample_data=pd.concat([df[df['Churn']==0].sample(1587),df[df['Churn']==1]])
X=sample_data.drop('Churn', axis=1)
y=sample_data['Churn']

In [ ]:
y.value_counts()

#### Train test split

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=42)

In [ ]:
X_train.describe()

In [ ]:
X_test.describe()

In [ ]:
print(f"Train target variable: {y_train.value_counts()}")
print(f"Train target variable: {y_test.value_counts()}")

#### Standard Scaler

In [ ]:
from sklearn.preprocessing import StandardScaler
scale = StandardScaler()
X_train = scale.fit_transform(X_train[int_feat])
X_test = scale.transform(X_test[int_feat])

## Model training, testing and tuning: 

### Decision Tree model - 
***Lets us try to use basic Decision Tree model to check the accuracy based on the dataset***

In [ ]:
# Decision Tree model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
dt_model = DecisionTreeClassifier(criterion = 'entropy' )
dt_model.fit(X_train, y_train)
print(f"Accuracy score for Train data: {dt_model.score(X_train , y_train)}")
test_pred = dt_model.predict(X_test)
print(f"Accuracy Score for Test data: {accuracy_score(y_test, test_pred)}")

In [ ]:
print(classification_report(y_test, test_pred))

Here the accuracy score for the Train data is high whic is aroung 100%, where in Test data is low - around 66%. which means the model is overfitted on Training data.  
This is expected in terms of Tree based modeling as it tend to overfit on the Trianing data. 
So let us try to use various techiniques to find which works best for this dataset

In [ ]:
# Ensemble Techniques - Basic Models with weak classifiers
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier,  GradientBoostingClassifier, RandomForestClassifier
models = [RandomForestClassifier(), AdaBoostClassifier(), AdaBoostClassifier(base_estimator=dt_model),BaggingClassifier(), GradientBoostingClassifier()]
for model in models:
    model.fit(X_train,y_train)
    print(f"{model} : Results\n")
    print(f"Accuracy score for Train data: {model.score(X_train , y_train)}")
    test_pred = model.predict(X_test)
    print(f"Accuracy Score for Test data: {accuracy_score(y_test, test_pred)}\n")
    print(f"Classification report \n {classification_report(y_test,test_pred)}\n")
    
    

***Bias and Variance comparision:***
GradeientBoostingClassifier seems to work better compared to other models. Bais & Variance of the model seems to be similar. other 3 models are seems to be overfitting. So, lets take GradientBoostingClassifier and fine tune with Hyperparameters.  
Also, the Recall and Precission values are more or less similar. f1 score is around 75% which tells us the Models works find on the dataset. Further fine tunining might help to improve the model accuracy.

### Hypertuning the Gradient boosting techinque

In [ ]:
n_estimators=[60,70,80,90,100,110,120,130,140,150]
for i in n_estimators:
    model = GradientBoostingClassifier(n_estimators=i, random_state=1)
    model.fit(X_train,y_train)
    print(f"{model} no of estimators:{i} : Results\n")
    print(f"Accuracy score for Train data: {model.score(X_train , y_train)}")
    test_pred = model.predict(X_test)
    print(f"Accuracy Score for Test data: {accuracy_score(y_test, test_pred)}\n")

#### Model with 100 estimators has high accuracy rate between both Training and test

In [ ]:
model = GradientBoostingClassifier(n_estimators=100, random_state=1)
model.fit(X_train,y_train)
print(f"{model} no of estimators:{100} : Results\n")
print(f"Accuracy score for Train data: {model.score(X_train , y_train)}")
test_pred = model.predict(X_test)
print(f"Accuracy Score for Test data: {accuracy_score(y_test, test_pred)}\n")

In [ ]:
print(classification_report(y_test, test_pred))

In [ ]:
sns.heatmap(confusion_matrix(y_test, test_pred), annot=True, fmt='2d')

***Observations:***
1. Model is giving us the accuracy score of around 80% on train data and 75% on test data. the output is comparitively good compared to other models. 
2. Precision and Recall score is good.
3. There are definetly room for improvment on the score by using Oversampling techinique

### Pickle the selected model for future use. 

In [ ]:
import pickle
pickle.dump(model, open('model.sav','wb'))

# GUI development